# AKANE E2-4：人がルールを書かなくても、AIが自分の言ったことを確かめられるか

前回（E2-3）は「インパラは哺乳類？」のような○×問題を、**人が書いたルール**でつないだ。
今回は、ルールのもとになる知識を **Gemma自身に聞く**。たとえば

- "Can something be both a mammal and a bird at the same time?"（哺乳類で、しかも鳥であるものはある？）
- "Can something be both a vegetable and a man-made object at the same time?"

Gemmaが「No（両立しない）」と答えた組だけを、ルールとして使う。

**このノートでやること**：Gemmaにルールの質問を314組×2回（順番を入れ替えて）聞き、答えをドライブに保存するだけ。
○×問題の答えは前回保存したもの（E2_results/wn_mouth_gemma-2-2b-it.json）を使うので、聞き直さない。
そのあとの計算（つながりで考える方法と、数える方法の比べ）は、保存された答えを使って手元のPCで行う。

**時間の目安**（T4 GPU）：5〜10分。計画は sims/e2_selfrule/PLAN.md（結果を見る前に固定）。

## 0. 準備
1. 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU**。
2. 鍵マークの `HF_TOKEN`（前と同じもの）にアクセスを許可。
3. 「ランタイム」→「すべてのセルを実行」。

In [ ]:
# ===== 設定 =====
USE_MOCK_MOUTH = False      # True: 作り物の口（動作確認用。結果は研究に使わない）
SAVE_TO_DRIVE  = True
MODEL_ID = "google/gemma-2-2b-it"
DATA_SEED = 0

In [ ]:
import os, json, time, itertools, collections
import numpy as np
for pkg, mod in [("python-sat", "pysat"), ("nltk", "nltk"), ("wordfreq", "wordfreq")]:
    try: __import__(mod)
    except ImportError: os.system(f"pip -q install {pkg}")
from pysat.formula import WCNF
from pysat.examples.rc2 import RC2
from sklearn.metrics import roc_auc_score

OUT = "/content/E2_results" if os.path.exists("/content") else "E2_results"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E2_results"
os.makedirs(OUT, exist_ok=True)
print("保存先:", OUT)

## 1. テストを作る（前回と同じもの）

In [ ]:
import nltk
from nltk.corpus import wordnet as wn
from wordfreq import zipf_frequency
try: nltk.data.find("corpora/wordnet")
except LookupError:
    try: nltk.data.find("corpora/wordnet.zip")
    except LookupError: nltk.download("wordnet", quiet=True)
# 大分類（7つ）と種類（25）。種類は WordNet の見出し、英語の言い方、大分類
CONT = {"animal": "animal", "plant": "plant", "food": "food", "artifact": "man-made object",
        "activity": "activity", "condition": "medical condition", "feeling": "feeling"}
CATS = {"mammal.n.01": ("mammal", "animal"), "bird.n.01": ("bird", "animal"), "fish.n.01": ("fish", "animal"),
        "insect.n.01": ("insect", "animal"), "reptile.n.01": ("reptile", "animal"),
        "tree.n.01": ("tree", "plant"), "flower.n.01": ("flower", "plant"),
        "vegetable.n.01": ("vegetable", "food"), "edible_fruit.n.01": ("fruit", "food"), "beverage.n.01": ("drink", "food"),
        "dish.n.02": ("dish", "food"), "baked_goods.n.01": ("baked good", "food"),
        "tool.n.01": ("tool", "artifact"), "vehicle.n.01": ("vehicle", "artifact"),
        "musical_instrument.n.01": ("musical instrument", "artifact"), "weapon.n.01": ("weapon", "artifact"),
        "furniture.n.01": ("furniture", "artifact"), "garment.n.01": ("clothing", "artifact"),
        "container.n.01": ("container", "artifact"), "building.n.01": ("building", "artifact"),
        "sport.n.01": ("sport", "activity"), "dance.n.01": ("dance", "activity"), "game.n.01": ("game", "activity"),
        "disease.n.01": ("disease", "condition"), "emotion.n.01": ("emotion", "feeling")}
# どちらとも言える大分類の組（偽の問題に使わない）：植物と食べ物、動物と食べ物
AMBIG = {frozenset(("plant", "food")), frozenset(("animal", "food"))}
MASS = {"furniture", "clothing"}
art = lambda w: w if w in MASS else ("an " if w[0] in "aeiou" else "a ") + w
lemma = lambda s: s.lemmas()[0].name()
by_co = collections.defaultdict(list)          # 種類 → ことば
for sid in CATS:
    S = wn.synset(sid)
    for leaf in S.closure(lambda s: s.hyponyms()):
        w = lemma(leaf)
        if "_" in w or not w.isalpha() or not w.islower() or zipf_frequency(w, "en") < 2.5: continue
        if len(wn.synsets(w)) != 1: continue        # 意味が1つしかない語だけ（取り違えを避ける）
        anc = set(leaf.closure(lambda s: s.hypernyms()))
        hits = [c for c in CATS if wn.synset(c) in anc]
        if hits == [sid]: by_co[sid].append(w)
rng = np.random.default_rng(DATA_SEED)
COUNTRIES = [k for k in CATS if len(set(by_co[k])) >= 4]
STMT, GOLD, KIND, CITY = [], [], [], []
idx = {}
def add(key, text, gold, kind, city=None):
    if key in idx: return idx[key]
    idx[key] = len(STMT); STMT.append(text); GOLD.append(gold); KIND.append(kind); CITY.append(city)
    return idx[key]
cont_of = {k: CATS[k][1] for k in COUNTRIES}
same_cont = collections.defaultdict(list)
for k in COUNTRIES: same_cont[cont_of[k]].append(k)
others = lambda z: [w for w in CONT if w != z and frozenset((w, z)) not in AMBIG]
# 種類→大分類
for k in COUNTRIES:
    z = cont_of[k]; fz = rng.choice(others(z))
    for zz in (z, fz):
        add(("CK", k, zz), f"{art(CATS[k][0]).capitalize()} is a kind of {CONT[zz]}.", int(zz == z), "種類→大分類")
# ことば→種類、ことば→大分類（1つの種類から多くても30語）
CITIES = []
for k in COUNTRIES:
    ws = sorted(set(by_co[k]))
    pick = list(rng.choice(ws, min(30, len(ws)), replace=False))
    for name in pick:
        z = cont_of[k]; CITIES.append(name)
        same = [w for w in same_cont[z] if w != k]     # 偽の種類は、なるべく同じ大分類から2つ
        pool = same if len(same) >= 2 else same + [w for w in COUNTRIES if cont_of[w] in others(z)]
        fks = list(rng.choice(pool, 2, replace=False))
        for kk in [k] + fks:
            add(("CC", name, kk), f"{name.capitalize()} is a kind of {CATS[kk][0]}.", int(kk == k), "ことば→種類", name)
        for zz in (z, rng.choice(others(z))):
            add(("CL", name, zz), f"{name.capitalize()} is a kind of {CONT[zz]}.", int(zz == z), "ことば→大分類", name)
GOLD = np.array(GOLD)
DUMMY = len(STMT)   # 「両立しない」ルールの結論に使う、いつも0の素子
# ルール：(前提1, 前提2, 結論)。結論がDUMMYなら「前提が両方1ではいけない」
RULES = set()
for (t, a, b), i in list(idx.items()):
    if t == "CC":
        for zz in CONT:
            if ("CK", b, zz) in idx and ("CL", a, zz) in idx:
                RULES.add((i, idx[("CK", b, zz)], idx[("CL", a, zz)]))     # ことばが種類に入り、種類が大分類に入る → ことばはその大分類
def excl(keys):
    ids = [idx[k] for k in keys if k in idx]
    for p, q in itertools.combinations(sorted(ids), 2): RULES.add((p, q, DUMMY))
for name in set(CITIES):
    excl([("CC", name, k) for k in COUNTRIES]); excl([("CL", name, z) for z in CONT])
for k in COUNTRIES:
    excl([("CK", k, z) for z in CONT])
RULES = sorted(RULES)
# ルールが正解で全部守られているか（守られていなければデータの誤り）
G = np.append(GOLD, 0)
bad = sum(1 for p, q, h in RULES if G[p] == 1 and G[q] == 1 and G[h] == 0)
print(f"種類 {len(COUNTRIES)}、ことば {len(set(CITIES))}、問題 {len(STMT)}（本物 {GOLD.sum()}）、ルール {len(RULES)}本、正解が破るルール {bad}")
print("例:", " | ".join(STMT[i] for i in [0, 1, 60, 61, 62, 63, 64]))

## 2. 前回の○×問題の答えを読み込む（聞き直さない）

In [ ]:
def mock_mouth(y, rng):
    lg = 2.0 * (2 * y - 1) + rng.normal(0, 1.5, len(y))
    flip = rng.random(len(y)) < 0.12
    lg[flip] = -np.sign(lg[flip]) * np.abs(rng.normal(2, 0.7, flip.sum()))
    return lg

cache = f"{OUT}/wn_mouth_{'MOCK' if USE_MOCK_MOUTH else MODEL_ID.split('/')[-1]}.json"
if os.path.exists(cache):
    d = json.load(open(cache))
    if d["stmt"] != STMT: raise SystemExit("保存済みの答えと問題が合わない（ライブラリの版の違いなど）。ファイルを消してやり直す")
    LOGIT = np.array(d["logit"]); print("保存済みの答えを読み込んだ:", cache)
elif USE_MOCK_MOUTH:
    LOGIT = mock_mouth(GOLD, np.random.default_rng(123)); json.dump({"stmt": STMT, "logit": LOGIT.tolist()}, open(cache, "w"))
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from google.colab import userdata
    tok = AutoTokenizer.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"),
                                                 torch_dtype=torch.float16, device_map="cuda").eval()
    YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]
    def prompt(s):
        msg = [{"role": "user", "content": f"Is the following statement true? Answer Yes or No only.\nStatement: {s}"}]
        return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    LOGIT = np.zeros(len(STMT)); B = 64; t = time.time()
    for i in range(0, len(STMT), B):
        enc = tok([prompt(s) for s in STMT[i:i + B]], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = model(**enc).logits[:, -1, :].float()
        LOGIT[i:i + B] = (last[:, YES] - last[:, NO]).cpu().numpy()
        if (i // B) % 20 == 0: print(f"{i + B}/{len(STMT)}  {time.time() - t:.0f}秒")
    json.dump({"stmt": STMT, "logit": LOGIT.tolist()}, open(cache, "w")); print("保存:", cache)
for k in ["ことば→種類", "ことば→大分類", "種類→大分類"]:
    s = np.array([x == k for x in KIND])
    print(f"口だけの正解率 {k}: {((LOGIT[s] > 0) == GOLD[s]).mean():.3f}")

## 3. ルールの質問の組を作る
同じことばの問題に一緒に出てくるラベルの組（種類と種類、大分類と大分類、種類と大分類）。
正しい答え（WordNet）では、「種類」と「その種類の大分類」（例：哺乳類と動物）だけが両立する。

In [ ]:
PAIRS = set()
for name in set(CITIES):
    L = [("K", k) for k in COUNTRIES if ("CC", name, k) in idx] + [("Z", z) for z in CONT if ("CL", name, z) in idx]
    for p, q in itertools.combinations(L, 2): PAIRS.add(tuple(sorted((p, q))))
for k in COUNTRIES:
    for p, q in itertools.combinations([("Z", z) for z in CONT if ("CK", k, z) in idx], 2): PAIRS.add(tuple(sorted((p, q))))
PAIRS = sorted(PAIRS)
def truth_excl(p, q):                      # True = 両立しない（WordNet）
    if p[0] == q[0]: return True
    k, z = (p, q) if p[0] == "K" else (q, p)
    return cont_of[k[1]] != z[1]
MASS2 = {"furniture", "clothing", "food"}
def name_of(lab):
    w = CATS[lab[1]][0] if lab[0] == "K" else CONT[lab[1]]
    return w if w in MASS2 else ("an " if w[0] in "aeiou" else "a ") + w
RQ = [f"Can something be both {name_of(p)} and {name_of(q)} at the same time? Answer Yes or No only." for p, q in PAIRS] + \
     [f"Can something be both {name_of(q)} and {name_of(p)} at the same time? Answer Yes or No only." for p, q in PAIRS]
TRUTH = [truth_excl(p, q) for p, q in PAIRS]
print(f"組 {len(PAIRS)}（両立しない {sum(TRUTH)}、両立する {len(TRUTH) - sum(TRUTH)}）、質問 {len(RQ)}")
print("例:", RQ[0], "|", RQ[len(PAIRS) // 2])

## 4. Gemmaにルールを聞く（保存するので2回目からは飛ばせる）

In [ ]:
rcache = f"{OUT}/E2-4_rules_{'MOCK' if USE_MOCK_MOUTH else MODEL_ID.split('/')[-1]}.json"
if os.path.exists(rcache):
    d = json.load(open(rcache))
    if d["questions"] != RQ: raise SystemExit("保存済みのルールの答えと質問が合わない。ファイルを消してやり直す")
    RLOGIT = np.array(d["logit"]); print("保存済みの答えを読み込んだ:", rcache)
elif USE_MOCK_MOUTH:
    r = np.random.default_rng(7); t = np.array(TRUTH * 2)
    RLOGIT = np.where(t, -2.0, 2.0) + r.normal(0, 2.0, len(RQ))
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from google.colab import userdata
    if "model" not in globals():
        tok = AutoTokenizer.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"),
                                                     torch_dtype=torch.float16, device_map="cuda").eval()
    YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]
    def rprompt(s):
        return tok.apply_chat_template([{"role": "user", "content": s}], tokenize=False, add_generation_prompt=True)
    RLOGIT = np.zeros(len(RQ)); B = 64; t = time.time()
    for i in range(0, len(RQ), B):
        enc = tok([rprompt(s) for s in RQ[i:i + B]], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = model(**enc).logits[:, -1, :].float()
        RLOGIT[i:i + B] = (last[:, YES] - last[:, NO]).cpu().numpy()
    print(f"{len(RQ)}問 {time.time() - t:.0f}秒")
if not os.path.exists(rcache):
    json.dump({"model": "MOCK" if USE_MOCK_MOUTH else MODEL_ID, "pairs": PAIRS, "questions": RQ,
               "logit": RLOGIT.tolist(), "truth_excl": TRUTH}, open(rcache, "w"), ensure_ascii=False)
    print("保存:", rcache)

## 5. Gemmaのルールの当たり具合（ここまでで終わり。結果のファイルを手元で読みます）

In [ ]:
n = len(PAIRS); avg = (RLOGIT[:n] + RLOGIT[n:]) / 2
g_excl = avg < 0; T = np.array(TRUTH)
print(f"Gemmaのルール：正しい判断 {(g_excl == T).mean():.1%}（{(g_excl == T).sum()}/{n}組）")
print(f"  見逃し（本当は両立しないのに、両立すると答えた）: {(T & ~g_excl).sum()} / {T.sum()}")
print(f"  間違ったルール（本当は両立するのに、両立しないと答えた）: {(~T & g_excl).sum()} / {(~T).sum()}")
print(f"  順番を入れ替えると答えが変わった組: {((RLOGIT[:n] < 0) != (RLOGIT[n:] < 0)).sum()}")
print("\n間違ったルールの例:")
for i in np.nonzero(~T & g_excl)[0][:10]: print("  ", RQ[i])
print("見逃しの例:")
for i in np.nonzero(T & ~g_excl)[0][:10]: print("  ", RQ[i])